# Accent transforms

In [1]:
from pathlib import Path
import os
import sys
import soundfile as sf
import pandas as pd

ROOT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'raw-data' / 'accent').is_dir())
sys.path.insert(0, str(ROOT_DIR / 'utilities'))
os.chdir(ROOT_DIR / 'utilities')
from transform_audio import transform_list, fft_file, stft_file, cwt_file, erblet_file

data_dir = ROOT_DIR / 'raw-data' / 'accent' / 'full-accent-audio-processed-trimmed'
OUTPUT_DIR = ROOT_DIR / 'raw-data' / 'accent' / 'transforms'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metadata_df = pd.read_csv(ROOT_DIR / 'raw-data/accent/full-accent.csv')
file_names = metadata_df['processed_filename'].tolist()
file_list = [data_dir / name for name in file_names]
os.chdir(OUTPUT_DIR)
print(f'{len(file_list)} recordings. Output: {OUTPUT_DIR}')

1678 recordings. Output: /Users/jiaxin/Documents/Summer 2025/hierarchical-bayesian-model-validation/raw-data/accent/transforms


## FFT

In [2]:
max_n = max(sf.info(path).frames for path in file_list)
transform_list(fft_file, file_list, file_names, n=max_n)

Computing and exporting coefficients:   0%|          | 0/1678 [00:00<?, ?it/s]

## STFT

In [3]:
transform_list(stft_file, file_list, file_names)

Computing and exporting coefficients:   0%|          | 0/1678 [00:00<?, ?it/s]

## CWT

In [2]:
transform_list(cwt_file, file_list, file_names,
               low_freq=10, high_freq=8000, num_scales=15)

Computing and exporting coefficients:   0%|          | 0/1678 [00:00<?, ?it/s]

## ERBlet

In [ ]:
transform_list(erblet_file, file_list, file_names)

In [2]:
import transform_audio as ta
import matlab.engine

utilities_dir = Path(ta.__file__).resolve().parent
ltfat_dir = (utilities_dir / 'ltfat_path.txt').read_text().strip()

ta.eng = matlab.engine.start_matlab()
ta.eng.addpath(str(utilities_dir), ltfat_dir, nargout=0)
ta.eng.ltfatstart(0.0, 'nojava', nargout=0)
ta.USE_MATLAB = True

transform_list(
    ta.erblet_file,
    [str(path) for path in file_list],
    file_names
)


Computing and exporting coefficients:   0%|          | 0/1678 [00:00<?, ?it/s]